In [2]:
# 04_user_logs_features
#
# 목적: user_logs.csv(30GB, 3억 9천만 행)를 유저(msno) 단위로 다중 윈도우
#      (최근 7/30/90일 + 전체 히스토리, 컷오프 2017-02-28) 집계한다.
#
# 방식: 컷오프 기준 경과일수(days_before)를 5개 구간(bucket)으로 한 번만 나눠
#      청크마다 단일 groupby로 부분집계한 뒤, 필요한 윈도우는 해당 구간들을 합산해서 구성한다.
#      (윈도우마다 별도로 groupby를 반복하면 청크당 연산이 배로 늘어나므로 이렇게 설계함)
#        b0_6    : 0~6일 전  (최근 7일)
#        b7_14   : 7~14일 전 (b0_6과 합치면 최근 15일 = trend의 "recent15")
#        b15_29  : 15~29일 전 (trend의 "prior15", b0_6+b7_14+b15_29 = 최근 30일)
#        b30_89  : 30~89일 전 (위 합치면 최근 90일)
#        b90_plus: 90일 이전 전체 (위 합치면 전체 히스토리)
#
# user_logs.csv 컬럼 설명
# msno       : 유저 ID (1유저 다건 로그)
# date       : 로그 날짜
# num_25/50/75/985/100 : 재생 완료율 구간별 곡 수
# num_unq    : 고유 곡 수
# total_secs : 총 재생 시간(초), 이상치 존재 -> 0~172800 범위만 유효로 간주
#
# 출력 피처 (윈도우별 {win} = d7/d30/d90/full):
#   {win}_active_days, {win}_avg_num_unq, {win}_avg_total_secs, {win}_completion_rate
#   trend_secs_recent15_vs_prior15 (최근 15일 평균 재생초 대비 그 이전 15일 평균의 변화율)
#
# 입력: data/raw/user_logs.csv
# 출력: data/processed/features_user_logs.csv
# 캐시: data/processed/user_logs_features_cache.pkl (30GB 전체 스캔은 오래 걸려서 재실행 시 재사용)

In [3]:
import time
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH = PROCESSED_DIR / "user_logs_features_cache.pkl"

CUTOFF_DATE = pd.Timestamp("2017-02-28")
TOTAL_SECS_CAP = 86400 * 2
CHUNK_SIZE = 10_000_000

BUCKET_BINS = [-1, 6, 14, 29, 89, 100_000]
BUCKET_LABELS = ["b0_6", "b7_14", "b15_29", "b30_89", "b90_plus"]

WINDOW_BUCKETS = {
    "d7": ["b0_6"],
    "d30": ["b0_6", "b7_14", "b15_29"],
    "d90": ["b0_6", "b7_14", "b15_29", "b30_89"],
    "full": ["b0_6", "b7_14", "b15_29", "b30_89", "b90_plus"],
}
METRICS = ["logdays", "num_unq_sum", "total_plays_sum", "num_100_sum", "secs_sum", "secs_n"]

In [4]:
def process_chunk(chunk):
    log_date = pd.to_datetime(chunk["date"], format="%Y%m%d")
    days_before = (CUTOFF_DATE - log_date).dt.days
    chunk = chunk.copy()
    chunk["bucket"] = pd.cut(days_before, bins=BUCKET_BINS, labels=BUCKET_LABELS)

    chunk["total_plays"] = chunk[["num_25", "num_50", "num_75", "num_985", "num_100"]].sum(axis=1)
    valid_secs_mask = (chunk["total_secs"] >= 0) & (chunk["total_secs"] <= TOTAL_SECS_CAP)
    chunk["total_secs_valid"] = chunk["total_secs"].where(valid_secs_mask)

    grouped = chunk.groupby(["msno", "bucket"], observed=True).agg(
        logdays=("msno", "size"),
        num_unq_sum=("num_unq", "sum"),
        total_plays_sum=("total_plays", "sum"),
        num_100_sum=("num_100", "sum"),
        secs_sum=("total_secs_valid", "sum"),
        secs_n=("total_secs_valid", "count"),
    )
    return grouped.unstack("bucket", fill_value=0)


print("process_chunk 정의 완료")

process_chunk 정의 완료


In [5]:
if CACHE_PATH.exists():
    with open(CACHE_PATH, "rb") as f:
        acc = pickle.load(f)
    print(f"캐시 로드 완료: {acc.shape} ({CACHE_PATH})")
else:
    t0 = time.time()
    reader = pd.read_csv(RAW_DIR / "user_logs.csv", chunksize=CHUNK_SIZE)
    acc = None
    n_rows = 0

    for i, chunk in enumerate(reader):
        n_rows += len(chunk)
        result = process_chunk(chunk)
        acc = result if acc is None else acc.add(result, fill_value=0)
        print(f"chunk {i + 1} 처리 완료, 누적 {n_rows:,} 행, 유저 수 {len(acc):,}, 경과 {time.time() - t0:.0f}s")

    print(f"\n전체 처리 완료: {n_rows:,} 행, 총 {time.time() - t0:.0f}초 소요")

    with open(CACHE_PATH, "wb") as f:
        pickle.dump(acc, f)
    print(f"캐시 저장 완료: {CACHE_PATH}")

chunk 1 처리 완료, 누적 10,000,000 행, 유저 수 1,617,125, 경과 16s
chunk 2 처리 완료, 누적 20,000,000 행, 유저 수 1,719,966, 경과 34s
chunk 3 처리 완료, 누적 30,000,000 행, 유저 수 1,822,947, 경과 52s
chunk 4 처리 완료, 누적 40,000,000 행, 유저 수 1,925,303, 경과 70s
chunk 5 처리 완료, 누적 50,000,000 행, 유저 수 2,027,722, 경과 87s
chunk 6 처리 완료, 누적 60,000,000 행, 유저 수 2,129,213, 경과 104s
chunk 7 처리 완료, 누적 70,000,000 행, 유저 수 2,231,289, 경과 122s
chunk 8 처리 완료, 누적 80,000,000 행, 유저 수 2,332,503, 경과 140s
chunk 9 처리 완료, 누적 90,000,000 행, 유저 수 2,434,213, 경과 157s
chunk 10 처리 완료, 누적 100,000,000 행, 유저 수 2,535,325, 경과 175s
chunk 11 처리 완료, 누적 110,000,000 행, 유저 수 2,636,263, 경과 193s
chunk 12 처리 완료, 누적 120,000,000 행, 유저 수 2,737,151, 경과 212s
chunk 13 처리 완료, 누적 130,000,000 행, 유저 수 2,837,716, 경과 230s
chunk 14 처리 완료, 누적 140,000,000 행, 유저 수 2,938,036, 경과 248s
chunk 15 처리 완료, 누적 150,000,000 행, 유저 수 3,037,975, 경과 267s
chunk 16 처리 완료, 누적 160,000,000 행, 유저 수 3,137,922, 경과 285s
chunk 17 처리 완료, 누적 170,000,000 행, 유저 수 3,237,447, 경과 304s
chunk 18 처리 완료, 누적 180,000,000 행, 유저 

In [6]:
def window_sum(acc, metric, buckets):
    cols = [(metric, b) for b in buckets if (metric, b) in acc.columns]
    return acc[cols].sum(axis=1)


features = pd.DataFrame(index=acc.index)

for win, buckets in WINDOW_BUCKETS.items():
    logdays = window_sum(acc, "logdays", buckets)
    num_unq_sum = window_sum(acc, "num_unq_sum", buckets)
    total_plays_sum = window_sum(acc, "total_plays_sum", buckets)
    num_100_sum = window_sum(acc, "num_100_sum", buckets)
    secs_sum = window_sum(acc, "secs_sum", buckets)
    secs_n = window_sum(acc, "secs_n", buckets)

    features[f"{win}_active_days"] = logdays
    features[f"{win}_avg_num_unq"] = (num_unq_sum / logdays).replace([np.inf, -np.inf], np.nan)
    features[f"{win}_avg_total_secs"] = (secs_sum / secs_n).replace([np.inf, -np.inf], np.nan)
    features[f"{win}_completion_rate"] = (num_100_sum / total_plays_sum).replace([np.inf, -np.inf], np.nan)

features.shape

(5234111, 16)

In [7]:
# 추세 신호: 최근 15일(b0_6+b7_14) 평균 재생초 vs 그 이전 15일(b15_29) 평균 재생초 변화율
recent15_secs_sum = window_sum(acc, "secs_sum", ["b0_6", "b7_14"])
recent15_secs_n = window_sum(acc, "secs_n", ["b0_6", "b7_14"])
prior15_secs_sum = window_sum(acc, "secs_sum", ["b15_29"])
prior15_secs_n = window_sum(acc, "secs_n", ["b15_29"])

recent15_avg = recent15_secs_sum / recent15_secs_n
prior15_avg = (prior15_secs_sum / prior15_secs_n).replace(0, np.nan)

features["trend_secs_recent15_vs_prior15"] = ((recent15_avg - prior15_avg) / prior15_avg).replace([np.inf, -np.inf], np.nan)

# 주의: prior15_avg가 0에 가까운 유저(15~29일 전 청취량이 거의 없던 유저)는 분모가 작아
# 변화율이 수만 %대로 폭주할 수 있다 (아래 describe의 max가 41,023 = 4,100,000%대).
# 트리 기반 모델은 이런 극단값에 비교적 강건해 그대로 뒀지만, 이 피처를 거리 기반/선형 모델에
# 직접 쓰거나 다른 용도로 재사용할 때는 winsorize/클리핑을 고려할 것.
print(features["trend_secs_recent15_vs_prior15"].describe())

count    899752.000000
mean          1.270498
std          82.899858
min          -0.999999
25%          -0.303177
50%          -0.003286
75%           0.408612
max       41023.179337
Name: trend_secs_recent15_vs_prior15, dtype: float64


In [8]:
features_user_logs = features.reset_index().rename(columns={"index": "msno"})

print(features_user_logs.shape)
print(features_user_logs.isna().sum())
features_user_logs.head()

(5234111, 18)
msno                                    0
d7_active_days                          0
d7_avg_num_unq                    4314136
d7_avg_total_secs                 4314145
d7_completion_rate                4314136
d30_active_days                         0
d30_avg_num_unq                   4105262
d30_avg_total_secs                4105274
d30_completion_rate               4105262
d90_active_days                         0
d90_avg_num_unq                   3756149
d90_avg_total_secs                3756158
d90_completion_rate               3756149
full_active_days                        0
full_avg_num_unq                        0
full_avg_total_secs                    58
full_completion_rate                    0
trend_secs_recent15_vs_prior15    4334359
dtype: int64


,msno,d7_active_days,d7_avg_num_unq,d7_avg_total_secs,d7_completion_rate,d30_active_days,d30_avg_num_unq,d30_avg_total_secs,d30_completion_rate,d90_active_days,d90_avg_num_unq,d90_avg_total_secs,d90_completion_rate,full_active_days,full_avg_num_unq,full_avg_total_secs,full_completion_rate,trend_secs_recent15_vs_prior15
0,+++4vcS9aMH7KWdfh5git6nA5fC5jjisd5H/NcM++WM=,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,1.0,2.000000,97.411000,0.000000,NaN
1,+++EI4HgyhgcJHIPXk/VRP7bt17+2joG39T6oEfJ+tc=,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,1.0,1.000000,56.868000,0.000000,NaN
2,+++FOrTS7ab3tIgIh8eWwX4FqRv8w/FoiOuyXsFvphY=,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,7.0,25.571429,7142.395857,0.647727,NaN
3,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,6.0,50.333333,12896.794167,0.969512,18.0,41.5,12253.563722,0.954887,66.0,79.969697,19916.705712,0.963135,622.0,86.180064,23576.915444,0.975543,3.898623
4,+++TipL0Kt3JvgNE9ahuJ8o+drJAnQINtxD4c5GePXI=,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,1.0,22.000000,3661.527000,0.608696,NaN


In [9]:
features_user_logs.to_csv(PROCESSED_DIR / "features_user_logs.csv", index=False)
print(f"저장 완료: {PROCESSED_DIR / 'features_user_logs.csv'} ({len(features_user_logs):,} rows, {features_user_logs['msno'].nunique():,} unique users)")

저장 완료: ..\data\processed\features_user_logs.csv (5,234,111 rows, 5,234,111 unique users)
